# 08 — Final Report: Pandas, Polars, and Dask End-to-End Benchmark

Notebook này tổng hợp kết quả từ các notebook benchmark trước đó và đọc trực tiếp các bảng đã export trong `results/table/`. Mục tiêu là viết phần report cuối dựa trên kết quả đã có, không phân tích lại từ raw benchmark logs.

**Input chính:**

- `02_real_row_scaling.ipynb`
- `03_physical_scaling.ipynb`
- `04_synthetic_stress.ipynb`
- `05_real_env_comparison.ipynb`
- `06_workload_breakdown.ipynb`
- `07_lazy_vs_eager_polars.ipynb`
- `01c_validate_synthetic.ipynb`

**Cách đọc:** đây là practical end-to-end dataframe analytics benchmark, không phải pure engine benchmark. Runtime bao gồm đọc Parquet, execution, optimization/scheduling, memory allocation và final materialization qua `.collect()` hoặc `.compute()`.

In [10]:
from pathlib import Path
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() in {"benchmark", "benchmarks", "analysis", "report", "data_prep"}:
    PROJECT_ROOT = PROJECT_ROOT.parent.parent if PROJECT_ROOT.parent.name == "notebooks" else PROJECT_ROOT.parent

TABLE_DIR = PROJECT_ROOT / "results" / "table"
FIG_DIR = PROJECT_ROOT / "results" / "figures"

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

def read_table(name):
    path = TABLE_DIR / name
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

available_tables = sorted(p.name for p in TABLE_DIR.glob("*.csv"))
print(f"Project root: {PROJECT_ROOT}")
print(f"Tables loaded from: {TABLE_DIR}")
print(f"Number of exported tables: {len(available_tables)}")

Project root: d:\Polar vs Dask
Tables loaded from: d:\Polar vs Dask\results\table
Number of exported tables: 62


## Report Input Summary

Final report này đọc các processed summary tables đã được export từ các notebook trước trong thư mục `results/table/`. Notebook này không chạy benchmark workload và cũng không tính lại toàn bộ phân tích từ raw benchmark logs.

Report chỉ sử dụng các bảng summary này để tổng hợp kết quả giữa các benchmark groups:

- real row-scaling results
- physical scaling / memory-pressure results
- synthetic stress results
- synthetic validation results
- environment comparison results
- workload breakdown results
- Polars Lazy-vs-Eager results

Cách tổ chức này giúp final report tập trung vào phần diễn giải, so sánh kết quả, rút ra insight chính và đưa ra kết luận cuối cùng, thay vì lặp lại các bước phân tích chi tiết đã được thực hiện ở những notebook trước.

## Executive Summary

Benchmark này đánh giá Pandas, Polars và Dask như các dataframe analytics frameworks trong điều kiện sử dụng thực tế. Thay vì chỉ đo tốc độ của một operator riêng lẻ, benchmark đo toàn bộ quá trình end-to-end của workload, bao gồm Parquet scan/read, query execution, scheduling hoặc optimization, memory allocation và final result materialization.

Cách đo này phản ánh sát hơn cách người dùng thực sự làm việc với dữ liệu lớn trong Python. Khi chạy một workload như `filter`, `groupby`, `join` hoặc `pipeline`, thời gian thực thi không chỉ đến từ phép toán chính, mà còn bị ảnh hưởng bởi I/O, decoding, memory usage, execution strategy và kích thước output cuối cùng.

The main findings are:

- **Polars is the strongest single-machine framework overall in the real row-scaling benchmark.**  
  Polars cho thấy hiệu năng rất tốt trên real data khi row count tăng từ `1M` lên `10M` và `50M`. Framework này đặc biệt mạnh trong các workload cần xử lý nhanh trên một máy, nhất là khi dữ liệu đủ lớn để tận dụng parallel execution và query optimization.

- **Pandas remains useful for small and simple workloads, but scales poorly as row count and memory pressure increase.**  
  Pandas vẫn phù hợp cho dữ liệu nhỏ, phân tích nhanh, hoặc workflow đơn giản. Tuy nhiên, khi row count tăng lớn hoặc workload tạo áp lực bộ nhớ cao, Pandas thường kém ổn định hơn và khó cạnh tranh với Polars hoặc Dask.

- **Dask is competitive for some large groupby workloads and partitioned execution, but scheduling overhead makes it less consistently fast.**  
  Dask có lợi thế khi dữ liệu cần xử lý theo partition hoặc khi workload có thể hưởng lợi từ distributed/out-of-core execution. Tuy nhiên, scheduling overhead và chi phí `.compute()` khiến Dask không luôn nhanh nhất, đặc biệt với workload nhỏ hoặc trung bình.

- **Physical scaling and synthetic stress results show that memory pressure and data distribution can change framework behavior.**  
  Kết quả ở physical scaling và synthetic stress cho thấy framework ranking có thể thay đổi khi physical dataset size tăng, khi dữ liệu có skew nặng, hoặc khi cardinality của key rất cao. Vì vậy, kết luận từ real row-scaling không nên được áp dụng trực tiếp cho mọi điều kiện dữ liệu.

- **Polars Lazy is not always faster than Polars Eager.**  
  Lazy execution giúp nhiều nhất ở các workload lớn như `groupby` và `pipeline`, nơi optimizer có thể nhìn toàn bộ query plan, đẩy filter/projection sớm hơn và giảm intermediate materialization. Tuy nhiên, Eager vẫn tốt hơn hoặc đủ dùng cho `join` và các `filter` đơn giản. Điều này cho thấy Lazy không nên được chọn mặc định chỉ vì nó có optimizer.

- **Environment matters.**  
  Kết quả trên Windows local, Linux hoặc Colab có thể khác nhau do RAM, swap/pagefile, filesystem, multiprocessing behavior và runtime resource limits. Vì vậy, nếu các môi trường không chạy trên cùng phần cứng, kết quả nên được diễn giải là environment comparison, không phải pure OS comparison.

Overall, benchmark cho thấy không có framework nào thắng tuyệt đối trong mọi trường hợp. Lựa chọn thực tế phụ thuộc vào workload type, dataset size, memory pressure, execution environment và việc người dùng ưu tiên runtime, memory usage hay reliability.

## 1. Introduction

Báo cáo này so sánh **Pandas**, **Polars** và **Dask** vì đây là ba dataframe frameworks đại diện cho ba cách tiếp cận phổ biến trong Python dataframe analytics: in-memory eager execution, high-performance single-machine execution, và partitioned/larger-than-memory execution.

**Pandas** được dùng như baseline quan trọng nhất. Đây là framework quen thuộc, có API phổ biến và phù hợp với dữ liệu nhỏ đến vừa hoặc các workflow phân tích đơn giản. Tuy nhiên, Pandas chủ yếu hoạt động theo mô hình eager execution trên một máy và thường gặp giới hạn khi row count, intermediate data hoặc memory footprint tăng mạnh.

**Polars** đại diện cho hướng single-machine high-performance analytics hiện đại. Polars sử dụng execution engine tối ưu cho dữ liệu columnar và hỗ trợ cả eager mode lẫn lazy mode. Lazy execution cho phép optimizer áp dụng các kỹ thuật như projection pushdown, predicate pushdown và whole-plan optimization. Tuy nhiên, benchmark này không giả định Lazy luôn tốt hơn, mà kiểm tra trực tiếp xem Lazy có thật sự cải thiện end-to-end runtime và memory behavior trong từng workload hay không.

**Dask** đại diện cho hướng partitioned dataframe và larger-than-memory processing. Dask hữu ích khi dữ liệu được chia partition hoặc khi workload vượt quá khả năng xử lý trực tiếp của một dataframe in-memory duy nhất. Tuy nhiên, Dask cũng có thêm chi phí từ scheduling, task graph execution, partition coordination và final materialization. Vì vậy, Dask không được xem là mặc định nhanh hơn trong mọi trường hợp.

Benchmark sử dụng **large-scale tabular review data** vì loại dữ liệu này có nhiều đặc điểm thường gặp trong analytics thực tế: số dòng lớn, nhiều cột dạng string, rating distribution bị skew, user/product key có cardinality cao, text length có long-tail, và các workload tự nhiên như `filter`, `groupby`, `join`, và `pipeline`. Đây là kiểu dữ liệu khiến performance phụ thuộc không chỉ vào một operator riêng lẻ, mà còn vào toàn bộ quá trình đọc dữ liệu, tối ưu hóa, cấp phát bộ nhớ, thực thi và materialize kết quả.

Vì vậy, báo cáo này nên được hiểu là một **end-to-end dataframe analytics benchmark**. Mục tiêu chính là trả lời câu hỏi thực tế: khi chạy các tác vụ dataframe phổ biến trên dataset review lớn, framework nào hoàn thành nhanh hơn, dùng bộ nhớ như thế nào, scale ra sao khi dữ liệu tăng, và nhạy với môi trường chạy như thế nào.

Benchmark này không cố tách riêng pure operator speed hoặc pure engine time khỏi các yếu tố như I/O, scheduling, memory allocation, OS cache, swap/pagefile behavior, hoặc `.collect()` / `.compute()` materialization. Những yếu tố này được giữ lại trong phép đo vì chúng là một phần của trải nghiệm thực tế khi xử lý dữ liệu lớn bằng dataframe frameworks trong Python.

## 2. Dataset

Dataset chính của benchmark là **Real Amazon Reviews dataset**. Đây là nguồn dữ liệu dùng cho main benchmark, đặc biệt là real row-scaling và environment comparison, vì các kết luận chính cần dựa trên dữ liệu thật thay vì dữ liệu được tạo nhân tạo.

Real Amazon Reviews dataset phù hợp với mục tiêu benchmark vì nó có nhiều đặc điểm thường gặp trong dữ liệu tabular thực tế: số dòng lớn, nhiều cột dạng string, rating distribution bị lệch, user/product key có cardinality cao, text length có long-tail, và các quan hệ tự nhiên phù hợp cho các workload như `filter`, `groupby`, `join`, và `pipeline`.

Synthetic dataset chỉ được dùng cho các thí nghiệm có kiểm soát, nơi real data không đủ để cô lập biến cần kiểm tra. Cụ thể, synthetic data được dùng cho hai nhóm chính:

1. **Physical scaling / memory pressure:** tạo dataset theo kích thước vật lý như `5GB`, `10GB`, `20GB` để kiểm tra framework behavior khi memory footprint tăng.
2. **Synthetic stress:** tạo các tình huống khó cô lập bằng real data, ví dụ heavy skew, high cardinality, hoặc 100M real-like synthetic rows.

Synthetic data **không thay thế real data** trong kết luận chính. Nó được dùng như công cụ để stress test, kiểm soát primary variable, và quan sát framework behavior trong các điều kiện cực đoan hoặc có kiểm soát. Vì vậy, kết quả từ synthetic benchmark không nên được generalize trực tiếp thành kết luận về real-world performance thông thường.

Trước khi được dùng trong benchmark, synthetic data đã được validation trong notebook `01c_validate_synthetic.ipynb`. Quá trình validation kiểm tra các yếu tố như schema, row count, memory footprint, cardinality, rating distribution, text length distribution, helpful vote distribution, key skew và unique ratio.

Kết quả validation cho thấy synthetic data đủ phù hợp cho mục tiêu physical scaling và synthetic stress benchmark. Các synthetic profiles cũng thể hiện đúng mục tiêu thiết kế: profile skewed tạo áp lực từ key skew, profile high-unique tạo áp lực từ high cardinality, và profile real-like giữ cấu trúc gần với real data để kiểm tra khả năng scale ở quy mô lớn hơn.

In [11]:
# Synthetic validation evidence exported/defined for stress interpretation.
validation_requirements = read_table("synthetic_stress_validation_requirements.csv")
display(validation_requirements)

,requirement,reason
0,schema compatibility,Stress results must use comparable columns and...
1,row count,Scenario labels such as 10M and 100M must matc...
2,key cardinality,High-cardinality stress must be intentional an...
3,key skew / frequency behavior,Heavy-skew stress must show stronger skew than...
4,text length distribution,String memory footprint affects runtime and pe...
5,rating distribution,Filter selectivity depends on value distribution.
6,null ratio,Null handling can affect physical and executio...
7,memory footprint,Stress interpretation depends on practical mem...


### Nhận xét

Bảng này tóm tắt các yêu cầu chính cần kiểm tra trước khi sử dụng synthetic data cho benchmark. Mục tiêu của bước validation không phải là làm synthetic data giống real data tuyệt đối, mà là đảm bảo synthetic data đủ hợp lệ để phục vụ đúng vai trò của từng benchmark group.

`schema compatibility` là điều kiện quan trọng nhất, vì các framework cần chạy cùng workload trên các cột tương đương nhau. Nếu schema khác nhau, kết quả benchmark có thể phản ánh sự khác biệt trong dữ liệu đầu vào thay vì sự khác biệt giữa framework.

`row count` cần được kiểm tra để đảm bảo các nhãn như `10M` hoặc `100M` phản ánh đúng quy mô dữ liệu thực tế. Điều này đặc biệt quan trọng với synthetic stress benchmark, vì sai lệch row count sẽ làm sai hướng diễn giải về scalability.

`key cardinality` và `key skew / frequency behavior` là hai yếu tố ảnh hưởng mạnh đến `groupby` và `join`. High-cardinality stress phải thật sự tạo áp lực về số lượng unique keys, trong khi heavy-skew stress phải thể hiện phân phối key lệch rõ hơn real data. Nếu hai đặc điểm này không được xác nhận, synthetic stress result sẽ mất ý nghĩa.

`text length distribution`, `rating distribution`, `null ratio`, và `memory footprint` cũng cần được kiểm tra vì chúng ảnh hưởng trực tiếp đến runtime, filter selectivity, memory allocation và final materialization. Đặc biệt, text length và memory footprint có thể làm thay đổi peak memory rất mạnh dù row count giống nhau.

Tóm lại, bảng này đóng vai trò như checklist để đảm bảo synthetic data được dùng đúng mục đích: physical scaling dùng để kiểm tra memory pressure, còn synthetic stress dùng để kiểm tra edge cases như skew và high cardinality. Các kết quả synthetic chỉ đáng tin cậy khi các yêu cầu validation này được thỏa mãn.

## 3. Methodology

Benchmark được thiết kế theo bốn nhóm độc lập. Mỗi nhóm kiểm soát một **primary variable** khác nhau để tránh trộn lẫn các hiệu ứng đến từ row count, physical dataset size, stress condition và execution environment.

| Benchmark group | Primary variable | Main dataset | Objective |
|---|---:|---|---|
| Real Data Row Scaling | Row count | Real Amazon Reviews | Đo khả năng scaling khi dữ liệu tăng từ `1M` lên `10M` và `50M` rows |
| Physical Scaling / Memory Pressure | Physical size / memory footprint | Synthetic real-like | Đo hành vi của framework khi dataset tăng theo GB và tiến gần hoặc vượt giới hạn RAM |
| Synthetic Stress | Stress condition | Synthetic stress profiles | Đo robustness dưới các điều kiện cực đoan như heavy skew, high cardinality và 100M-scale synthetic data |
| OS / Environment Comparison | Execution environment | Real Amazon Reviews | Đo độ ổn định và sensitivity giữa Windows local, Linux hoặc Colab runtime |

Bốn nhóm benchmark này không được trộn vào một ranking duy nhất, vì chúng trả lời các câu hỏi khác nhau. `Real Data Row Scaling` trả lời câu hỏi framework scale như thế nào khi số dòng tăng trên real data. `Physical Scaling / Memory Pressure` trả lời câu hỏi framework phản ứng ra sao khi physical dataset size và memory footprint tăng. `Synthetic Stress` tập trung vào edge cases như heavy skew, high cardinality hoặc large synthetic scale. `OS / Environment Comparison` kiểm tra tính reproducibility và độ nhạy với môi trường chạy.

Việc tách riêng các nhóm benchmark giúp kết quả dễ diễn giải hơn. Ví dụ, `100M synthetic` không nên được xem là bước scaling tiếp theo trực tiếp của `50M real data`, vì hai nhóm này khác nhau về data source, physical footprint, data distribution và workload scope. Tương tự, kết quả giữa Windows local và Colab/Linux không nên được gọi là pure OS comparison nếu phần cứng, RAM, filesystem hoặc runtime policy khác nhau.

Benchmark này đo **end-to-end dataframe workload performance**, không đo pure operator speed. Runtime của một workload bao gồm nhiều thành phần như Parquet scan/read, decoding, query execution, framework scheduling hoặc optimization, memory allocation và final result materialization.

Ví dụ, runtime của `filter` không chỉ đo chi phí evaluate predicate. Nó còn có thể bao gồm chi phí đọc Parquet, decode columns, apply predicate/projection pushdown nếu framework hỗ trợ, allocate output và materialize kết quả cuối cùng. Tương tự, runtime của `join` không chỉ phản ánh join algorithm, mà còn bị ảnh hưởng bởi key representation, build/probe cost, memory allocation, output size và materialization cost.

Do đó, các kết quả trong report này nên được hiểu là hiệu năng thực tế của toàn bộ dataframe workload trong Python, không phải tốc độ thuần của từng operator riêng lẻ.

**Timing and logging note.** Runtime trong các bảng kết quả được lấy từ timer nội bộ của từng workload, không phải từ khoảng cách timestamp giữa hai dòng log ngoài cùng trên terminal. Vì vậy, khoảng thời gian quan sát được trong terminal có thể lớn hơn `mean_time_s` hoặc runtime được ghi trong CSV.

Sự chênh lệch này là bình thường trong end-to-end benchmark, vì terminal timestamp có thể bao gồm nhiều phần không nằm trong timed region của workload, ví dụ: khởi động benchmark script, tạo scheduler/worker, warm-up run, cleanup giữa các runs, garbage collection, ghi log, ghi CSV output, shutdown cluster hoặc các bước orchestration của `run_pipeline.py`.

Điều này đặc biệt rõ với Dask. Dask thường in nhiều internal logs từ `distributed.scheduler`, `distributed.worker`, `distributed.nanny`, dashboard, worker registration, communication, cleanup hoặc cluster shutdown. Những log này phản ánh hoạt động của Dask distributed runtime và không nên được hiểu tự động là benchmark failure.

Một số warning như `Event loop was unresponsive` cho thấy scheduler hoặc nanny bị block trong vài giây, thường do long-running GIL-holding functions hoặc moving large chunks of data. Đây là tín hiệu về scheduling/runtime pressure và có thể giúp giải thích overhead hoặc instability. Tuy nhiên, nếu workload vẫn ghi đủ timed runs và result CSV có `status == "ok"`, thì run đó vẫn được tính là successful benchmark result.

Vì vậy, report ưu tiên sử dụng các metric đã được ghi trong result tables như `mean_runtime`, `mean_peak_memory`, `throughput`, `status`, và số successful runs. Terminal logs chỉ được dùng như supporting evidence để giải thích overhead, Dask scheduler behavior, cleanup cost hoặc environment sensitivity, không thay thế cho kết quả đã được aggregate.

## 4. Results — Real Data Row Scaling

`Real Data Row Scaling` là benchmark chính của report vì nó sử dụng **Real Amazon Reviews dataset** và kiểm tra khả năng scale khi row count tăng từ `1M` lên `10M` và `50M`. Nhóm benchmark này phản ánh gần nhất câu hỏi thực tế: khi dữ liệu thật ngày càng lớn, framework nào hoàn thành workload nhanh hơn, ổn định hơn và giữ throughput tốt hơn.

Kết quả tổng thể cho thấy **Polars** là nhóm framework có hiệu năng end-to-end tốt nhất trong phần lớn workload. Ở `10M`, `polars_eager` đứng đầu cả bốn workload gồm `filter`, `groupby`, `join`, và `pipeline`. Ở `50M`, kết quả phân hóa rõ hơn: `polars_lazy` đứng đầu ở `filter` và `pipeline`, `dask` đứng đầu ở `groupby`, còn `polars_eager` đứng đầu ở `join`.

Khi xét trung bình trên toàn bộ real row-scaling benchmark, `polars_lazy` có mean runtime thấp nhất, tiếp theo là `polars_eager`, sau đó là `dask`, và `pandas` là framework chậm nhất. Tuy nhiên, ranking tổng hợp chỉ nên được xem như overview. Ranking theo từng workload quan trọng hơn, vì mỗi workload có đặc điểm execution khác nhau và tạo áp lực khác nhau lên CPU, memory, I/O, scheduler và materialization.

Về workload cost, `join` là workload nặng nhất theo mean runtime, tiếp theo là `pipeline`, `filter`, rồi `groupby`. Kết quả này phù hợp với bản chất của workload. `Join` thường bị chi phối bởi key handling, join output size, memory allocation và final materialization. `Pipeline` gồm nhiều bước xử lý nên có thể khuếch đại chi phí đọc dữ liệu, intermediate data và output materialization. `Filter` đơn giản hơn về logic nhưng vẫn có thể tốn chi phí nếu output lớn. `GroupBy` có thể tạo output nhỏ hơn input nên trong một số trường hợp runtime thấp hơn các workload khác.

Về throughput, Polars thường giữ throughput cao hơn khi row count tăng, đặc biệt trong các workload phù hợp với columnar execution và query optimization. Dask có thể rất mạnh ở một số aggregation lớn, nổi bật là `groupby` ở `50M`, nhưng scheduling overhead và partition coordination khiến Dask không ổn định bằng Polars trên toàn bộ workload. Pandas vẫn phù hợp làm baseline và hữu ích với workload nhỏ hoặc đơn giản, nhưng khi row count tăng, runtime tăng mạnh và throughput giảm rõ rệt so với Polars và Dask.

Tóm lại, real row-scaling benchmark cho thấy Polars là lựa chọn mạnh nhất cho high-performance single-machine dataframe analytics trong phần lớn trường hợp. Tuy nhiên, không có framework nào thắng tuyệt đối ở mọi workload và mọi dataset size. `Dask` vẫn có lợi thế trong một số aggregation lớn, còn `Pandas` phù hợp hơn với quy mô nhỏ và workflow đơn giản.

In [12]:
real = read_table("real_row_scaling_summary.csv")
real["runtime_rank"] = real.groupby(["workload", "dataset_size"])["mean_time_s"].rank(method="min")

real_best = (
    real[real["runtime_rank"] == 1]
    [["dataset_size", "workload", "framework", "mean_time_s", "mean_peak_memory_gb", "mean_throughput_mrows_per_s"]]
    .sort_values(["dataset_size", "workload"])
)

display(real_best.round(3))

,dataset_size,workload,framework,mean_time_s,mean_peak_memory_gb,mean_throughput_mrows_per_s
18,10M,filter,polars_eager,1.793,4.320,5.584
22,10M,groupby,polars_eager,2.939,5.226,3.405
26,10M,join,polars_eager,3.808,5.507,2.667
30,10M,pipeline,polars_eager,4.977,5.372,2.013
2,1M,filter,polars_eager,0.317,0.746,3.155
7,1M,groupby,dask,0.318,0.704,3.153
10,1M,join,polars_eager,0.907,2.074,1.104
14,1M,pipeline,polars_eager,1.031,1.579,0.970
33,50M,filter,polars_lazy,30.594,12.151,1.636
39,50M,groupby,dask,11.238,4.585,4.453


### Nhận xét

Bảng này cho thấy framework tốt nhất thay đổi theo cả `dataset_size` và `workload`, nên không nên kết luận một framework thắng tuyệt đối trong mọi trường hợp. Tuy nhiên, Polars xuất hiện nhiều nhất trong nhóm kết quả tốt nhất, đặc biệt là `polars_eager` ở `1M` và `10M`, và `polars_lazy` ở một số workload lớn tại `50M`.

Ở `1M`, `polars_eager` đứng đầu ở `filter`, `join`, và `pipeline`, trong khi `dask` đứng đầu ở `groupby`. Điều này cho thấy với dataset nhỏ, overhead của Lazy hoặc Dask không phải lúc nào cũng có lợi, và Eager execution có thể là lựa chọn hiệu quả cho các workload đơn giản hoặc cần materialize kết quả nhanh.

Ở `10M`, `polars_eager` đứng đầu cả bốn workload. Đây là điểm mạnh rõ nhất của Polars Eager trong real row-scaling benchmark: nó giữ runtime thấp, throughput cao và memory usage ở mức hợp lý trên toàn bộ workload chính.

Ở `50M`, ranking bắt đầu phân hóa. `polars_lazy` đứng đầu ở `filter` và `pipeline`, cho thấy Lazy execution phát huy lợi thế khi dataset lớn hơn và optimizer có nhiều cơ hội giảm công việc hoặc tối ưu query plan. `dask` đứng đầu ở `groupby`, cho thấy partitioned execution có thể rất hiệu quả cho aggregation lớn. Trong khi đó, `polars_eager` vẫn đứng đầu ở `join`, cho thấy Eager là lựa chọn ổn định hơn cho join trong benchmark này.

Về throughput, các framework đứng đầu thường cũng có throughput cao hơn trong cùng workload. Tuy nhiên, throughput cần được đọc cùng runtime và memory, vì một framework có thể nhanh hơn nhưng dùng nhiều peak memory hơn, hoặc chỉ mạnh trong một workload cụ thể.

Tóm lại, bảng này củng cố kết luận rằng Polars là lựa chọn mạnh nhất tổng thể trong real row-scaling benchmark, nhưng kết quả theo workload vẫn quan trọng hơn ranking tổng hợp. `polars_eager` phù hợp nhất ở dataset nhỏ và trung bình, `polars_lazy` có lợi hơn ở một số workload lớn, còn `dask` có thể cạnh tranh mạnh ở aggregation lớn như `groupby 50M`.

In [13]:
real_framework_summary = (
    real.groupby("framework")
    .agg(
        mean_runtime_s=("mean_time_s", "mean"),
        mean_peak_memory_gb=("mean_peak_memory_gb", "mean"),
        mean_throughput_mrows_s=("mean_throughput_mrows_per_s", "mean"),
    )
    .sort_values("mean_runtime_s")
)

real_workload_cost = (
    real.groupby("workload")
    .agg(mean_runtime_s=("mean_time_s", "mean"), mean_peak_memory_gb=("mean_peak_memory_gb", "mean"))
    .sort_values("mean_runtime_s", ascending=False)
)

display(real_framework_summary.round(3))
display(real_workload_cost.round(3))

,mean_runtime_s,mean_peak_memory_gb,mean_throughput_mrows_s
framework,,,
polars_lazy,16.791,6.116,1.606
polars_eager,24.333,4.952,2.022
dask,35.915,5.366,1.380
pandas,308.645,6.722,0.222


,mean_runtime_s,mean_peak_memory_gb
workload,,
join,172.516,6.696
pipeline,89.618,5.931
filter,74.043,5.465
groupby,49.506,5.064


### Nhận xét

Hai bảng summary cho thấy bức tranh tổng quát của real row-scaling benchmark theo hai góc nhìn: theo framework và theo workload. Kết quả framework-level cho thấy `polars_lazy` có mean runtime thấp nhất, khoảng `16.79s`, tiếp theo là `polars_eager` với khoảng `24.33s`, `dask` với khoảng `35.92s`, và `pandas` chậm nhất với khoảng `308.65s`.

Điều này cho thấy Polars là nhóm framework mạnh nhất về runtime tổng thể trong benchmark này. `polars_lazy` có lợi thế runtime trung bình tốt nhất, nhưng `polars_eager` lại có throughput trung bình cao hơn và peak memory thấp hơn. Vì vậy, Lazy không nên được hiểu là tốt hơn toàn diện; nó mạnh về runtime tổng hợp, còn Eager vẫn rất cạnh tranh về throughput và memory.

`Dask` có runtime trung bình cao hơn Polars nhưng vẫn nhanh hơn Pandas rất nhiều. Điều này phản ánh đặc điểm của Dask: có thể hữu ích cho partitioned execution và một số workload lớn, nhưng scheduling overhead khiến hiệu năng trung bình không ổn định bằng Polars trong benchmark này. `Pandas` có runtime cao nhất và throughput thấp nhất, cho thấy giới hạn rõ khi dữ liệu tăng lớn.

Bảng workload-level cho thấy `join` là workload nặng nhất, với mean runtime khoảng `172.52s` và peak memory khoảng `6.70GB`. Điều này phù hợp với bản chất của join, vì workload này thường cần xử lý key, tạo output lớn, cấp phát bộ nhớ nhiều và materialize kết quả cuối cùng.

`Pipeline` là workload nặng thứ hai, với mean runtime khoảng `89.62s`. Vì pipeline gồm nhiều bước xử lý, chi phí có thể đến từ scan/read, intermediate results, optimization, memory allocation và final materialization. `Filter` đứng sau pipeline, còn `groupby` có mean runtime thấp nhất trong bốn workload, khoảng `49.51s`.

Tóm lại, kết quả tổng hợp cho thấy Polars là lựa chọn mạnh nhất về hiệu năng end-to-end trên real row-scaling benchmark, trong khi Pandas kém phù hợp khi dữ liệu lớn. Ở góc nhìn workload, `join` và `pipeline` là hai workload tạo áp lực lớn nhất, nên cần được xem xét kỹ khi đánh giá runtime, memory và khả năng scale của framework.

## 5. Results — Physical Scaling / Memory Pressure

`Physical Scaling / Memory Pressure` khác với `Real Data Row Scaling` vì primary variable ở đây là **physical dataset size** và memory footprint, không phải số dòng. Do đó, phần này được dùng để đánh giá framework behavior khi dataset tăng theo kích thước vật lý, memory pressure tăng, và workload tiến gần hoặc vượt giới hạn RAM của môi trường chạy.

Kết quả physical scaling cho thấy `polars_eager` có runtime tốt nhất trong nhiều workload, đặc biệt ở `filter`, `join`, và `pipeline` tại các mức `5GB` và `10GB`. Điều này cho thấy eager execution vẫn rất hiệu quả khi workload có thể được xử lý trực tiếp và chi phí lazy planning chưa tạo đủ lợi thế.

`Dask` dẫn đầu ở `groupby` trên cả `5GB`, `10GB`, và `20GB`, cho thấy partitioned aggregation có thể rất có lợi trong workload này. Khi aggregation có thể chia nhỏ theo partition và output cuối nhỏ hơn input, Dask có thể tận dụng mô hình distributed/partitioned execution tốt hơn so với các workload cần materialize output lớn.

Ở `20GB pipeline`, `polars_lazy` là framework dẫn đầu và dùng memory thấp hơn đáng kể. Kết quả này cho thấy Lazy planning có thể phát huy lợi thế trong pipeline nhiều bước, đặc biệt khi optimizer có thể giảm intermediate materialization hoặc tối ưu thứ tự thực thi trước khi collect kết quả cuối cùng.

Về memory pressure, `polars_lazy` thường có mean peak memory thấp hơn `polars_eager` trong physical scaling, trong khi `pandas` có mean memory cao nhất và runtime cao nhất. Điều này cho thấy Pandas kém phù hợp hơn khi physical dataset size tăng lớn. `Dask` không luôn nhanh nhất, nhưng vẫn có ý nghĩa trong các workload partitioned hoặc out-of-core, đặc biệt khi workload có thể chia nhỏ tốt như `groupby`.

Không có failure được ghi nhận trong exported failure table của physical scaling, nhưng degradation về runtime vẫn rất rõ, đặc biệt với Pandas ở `20GB`. Điều này cho thấy một workload có thể hoàn thành thành công nhưng vẫn không thực tế về mặt thời gian hoặc chi phí tài nguyên.

Một điểm quan trọng là nếu dataset lớn hơn RAM nhưng workload vẫn hoàn thành, điều đó không có nghĩa toàn bộ dataset được giữ trong RAM cùng lúc. Kết quả chỉ cho thấy end-to-end workload có thể hoàn thành trong môi trường chạy cụ thể. Việc hoàn thành có thể đến từ streaming, partitioning, OS page cache, pagefile/swap, hoặc vì output cuối nhỏ hơn input.

Tóm lại, physical scaling benchmark cho thấy framework behavior thay đổi rõ khi physical dataset size và memory pressure tăng. `Polars Eager` mạnh ở nhiều workload vừa và lớn, `Dask` nổi bật ở partitioned aggregation, `Polars Lazy` có lợi trong một số pipeline lớn, còn `Pandas` chịu degradation rõ nhất khi kích thước vật lý tăng.

In [14]:
physical = read_table("physical_scaling_summary.csv")
physical["runtime_rank"] = physical.groupby(["workload", "dataset_size"])["mean_time_s"].rank(method="min")

physical_best = (
    physical[physical["runtime_rank"] == 1]
    [["dataset_size", "workload", "framework", "mean_time_s", "mean_peak_memory_gb"]]
    .sort_values(["dataset_size", "workload"])
)
physical_framework_summary = (
    physical.groupby("framework")
    .agg(mean_runtime_s=("mean_time_s", "mean"), mean_peak_memory_gb=("mean_peak_memory_gb", "mean"))
    .sort_values("mean_runtime_s")
)

display(physical_best.round(3))
display(physical_framework_summary.round(3))

,dataset_size,workload,framework,mean_time_s,mean_peak_memory_gb
18,10GB,filter,polars_eager,2.753,6.386
23,10GB,groupby,dask,1.895,4.141
26,10GB,join,polars_eager,3.428,7.522
30,10GB,pipeline,polars_eager,4.822,7.924
34,20GB,filter,polars_eager,8.020,11.757
39,20GB,groupby,dask,3.759,5.385
42,20GB,join,polars_eager,11.129,12.081
45,20GB,pipeline,polars_lazy,8.443,4.613
2,5GB,filter,polars_eager,1.153,3.210
7,5GB,groupby,dask,1.065,2.307


,mean_runtime_s,mean_peak_memory_gb
framework,,
polars_eager,6.612,7.499
polars_lazy,7.448,5.166
dask,24.779,6.706
pandas,103.660,8.725


### Nhận xét

Hai bảng physical scaling cho thấy khi dataset tăng theo kích thước vật lý từ `5GB` lên `20GB`, framework ranking thay đổi theo workload. `polars_eager` là framework dẫn đầu nhiều nhất về runtime, đặc biệt ở `filter`, `join`, và `pipeline` tại `5GB` và `10GB`, cũng như `filter` và `join` tại `20GB`.

Ở `groupby`, `dask` đứng đầu ở cả ba mức `5GB`, `10GB`, và `20GB`. Điều này cho thấy partitioned aggregation của Dask có lợi trong workload này, đặc biệt khi dữ liệu đủ lớn và output aggregation có thể nhỏ hơn input. Tuy nhiên, kết quả framework-level vẫn cho thấy Dask không phải framework nhanh nhất trung bình, vì overhead của Dask ảnh hưởng nhiều ở các workload khác.

Ở `20GB pipeline`, `polars_lazy` là framework nhanh nhất với runtime khoảng `8.44s` và peak memory khoảng `4.61GB`. Đây là một kết quả quan trọng vì nó cho thấy Lazy execution có thể phát huy lợi thế khi pipeline lớn hơn và memory pressure cao hơn. Trong trường hợp này, Lazy không chỉ nhanh nhất mà còn dùng ít memory hơn đáng kể so với các kết quả Polars Eager ở các workload lớn khác.

Bảng framework-level cho thấy `polars_eager` có mean runtime thấp nhất, khoảng `6.61s`, nên là framework nhanh nhất trung bình trong physical scaling. `polars_lazy` đứng thứ hai về runtime, khoảng `7.45s`, nhưng lại có mean peak memory thấp nhất, khoảng `5.17GB`. Điều này cho thấy trade-off rõ ràng: Eager mạnh hơn về runtime trung bình, còn Lazy tốt hơn về memory pressure.

`Dask` có mean runtime cao hơn Polars, khoảng `24.78s`, nhưng vẫn có vai trò quan trọng ở `groupby`, nơi nó đứng đầu trên tất cả physical sizes. `Pandas` có mean runtime và mean peak memory cao nhất, lần lượt khoảng `103.66s` và `8.73GB`, cho thấy Pandas chịu ảnh hưởng mạnh nhất khi physical dataset size tăng.

Tóm lại, physical scaling benchmark cho thấy `polars_eager` là lựa chọn nhanh nhất trung bình, `polars_lazy` là lựa chọn tiết kiệm memory hơn và có lợi ở pipeline lớn, `dask` mạnh ở partitioned aggregation, còn `pandas` kém phù hợp nhất khi memory pressure tăng cao.

In [15]:
physical_failures = read_table("physical_scaling_failures.csv")
physical_over_ram = read_table("physical_scaling_over_ram_failures.csv")
print("physical_scaling_failures rows:", len(physical_failures))
print("physical_scaling_over_ram_failures rows:", len(physical_over_ram))
display(physical_failures)
display(physical_over_ram)

physical_scaling_failures rows: 0
physical_scaling_over_ram_failures rows: 0


,framework,workload,dataset_size,n_fail,statuses,notes


,timestamp,framework,workload,dataset_size,n_rows,run_index,time_s,peak_memory_mb,throughput_rows_per_s,status,notes,source_file,dataset_size_gb


### Nhận xét

Kết quả failure check cho thấy `physical_scaling_failures rows: 0` và `physical_scaling_over_ram_failures rows: 0`. Điều này nghĩa là trong các result files đã được load, không có failed run nào được ghi nhận cho nhóm physical scaling hoặc over-RAM physical scaling.

Đây là tín hiệu tích cực về mặt stability: các workload trong nhóm physical scaling đều hoàn thành thành công trong môi trường benchmark hiện tại. Vì vậy, phần phân tích có thể tập trung vào runtime degradation, peak memory và framework ranking, thay vì phải xử lý failure behavior.

Tuy nhiên, việc không có failed run không có nghĩa là mọi framework đều xử lý memory pressure tốt như nhau. Các bảng runtime và memory trước đó vẫn cho thấy sự khác biệt lớn giữa framework, đặc biệt là Pandas có runtime và memory cao hơn rõ rệt khi physical dataset size tăng. Nói cách khác, workload có thể hoàn thành thành công nhưng vẫn không thực tế nếu runtime quá cao hoặc memory usage quá lớn.

Tóm lại, physical scaling benchmark không ghi nhận failure, nhưng vẫn cho thấy memory pressure ảnh hưởng mạnh đến performance. Do đó, kết luận chính của section này nên tập trung vào degradation và resource efficiency, không phải failure rate.

## 6. Results — Synthetic Stress Benchmark

`Synthetic Stress Benchmark` gồm ba nhóm chính: **100M real-like synthetic**, **10M heavy skew**, và **10M high cardinality**. Mục tiêu của phần này là tìm robustness limits, failure modes và sensitivity của từng framework dưới các điều kiện dữ liệu cực đoan hoặc khó cô lập bằng real data. Kết quả synthetic stress không được dùng để thay thế kết luận chính từ real-data benchmark.

Ở **100M real-like synthetic**, reportable scope chỉ gồm `polars_lazy` và `dask` trên hai workload `filter` và `groupby`. Cả hai framework đều hoàn thành các workload này, nhưng `polars_lazy` nhanh hơn Dask trong cả hai trường hợp: khoảng `81.09s` so với `111.92s` ở `filter`, và khoảng `12.66s` so với `17.16s` ở `groupby`. Kết quả này cho thấy Polars Lazy có khả năng xử lý tốt các workload lớn khi workload không yêu cầu materialize output quá lớn.

Một số trường hợp 100M không được đưa vào reportable scope. `Pandas 100M` bị loại vì vượt practical memory envelope; `Polars Eager 100M` từng gây shutdown; còn `join` và `pipeline` ở 100M bị loại vì áp lực từ materialization và intermediate data quá lớn. Điều này cho thấy ở quy mô 100M, khả năng hoàn thành workload phụ thuộc mạnh vào execution strategy và kích thước output cuối cùng, không chỉ phụ thuộc vào tốc độ operator.

Ở **10M heavy skew**, `polars_eager` nhanh nhất ở `filter`, `join`, và `pipeline`, trong khi `polars_lazy` nhanh nhất ở `groupby`. Heavy skew làm lộ rõ workload sensitivity: aggregation trên hot keys có thể phản ứng rất khác so với join hoặc pipeline. Với `groupby`, Lazy có thể tận dụng execution strategy hoặc optimization tốt hơn trong trường hợp key distribution bị skew mạnh. Ngược lại, `join` và `pipeline` vẫn bị chi phối bởi key handling, output size, memory allocation và materialization cost.

Ở **10M high cardinality**, `polars_eager` nhanh nhất ở `filter`, `join`, và `pipeline`, còn `dask` nhanh nhất ở `groupby`. High cardinality làm tăng áp lực lên hash table, key handling và memory overhead, đặc biệt trong các workload như `groupby` và `join`. Kết quả này cho thấy high cardinality không ảnh hưởng đến mọi framework theo cùng một cách: `polars_eager` ổn định ở nhiều workload, `dask` có thể rất mạnh ở aggregation, còn Lazy không phải lúc nào cũng thắng khi key cardinality tăng cao.

Nhìn chung, synthetic stress benchmark cho thấy framework ranking có thể thay đổi đáng kể khi data distribution thay đổi. `Polars Eager` ổn định trong nhiều workload stress, `Polars Lazy` có lợi rõ trong một số aggregation hoặc large-scale workload, còn `Dask` có thể cạnh tranh mạnh ở `groupby` nhưng kém ổn định hơn ở `join` và `pipeline`.

Các kết quả này cần được đọc cùng caveat quan trọng: synthetic stress results không generalize trực tiếp thành kết luận chính về real-world performance thông thường. Chúng cho biết framework phản ứng thế nào dưới các điều kiện cực đoan đã kiểm soát, giúp bổ sung cho main benchmark bằng cách làm rõ robustness limits và failure modes.

In [16]:
stress_100m = read_table("synthetic_stress_100m_summary.csv")
stress_10m = read_table("synthetic_stress_10m_summary.csv")
stress_failures = read_table("synthetic_stress_failure_taxonomy.csv")

stress_10m["runtime_rank"] = stress_10m.groupby(["dataset_size", "workload"])["time_mean_s"].rank(method="min")
stress_10m_best = (
    stress_10m[stress_10m["runtime_rank"] == 1]
    [["dataset_size", "workload", "framework", "time_mean_s", "memory_gb", "rows_per_second_m"]]
    .sort_values(["dataset_size", "workload"])
)

display(stress_100m[["dataset_size", "framework", "workload", "time_mean_s", "memory_gb", "rows_per_second_m"]].round(3))
display(stress_10m_best.round(3))
display(stress_failures)

,dataset_size,framework,workload,time_mean_s,memory_gb,rows_per_second_m
0,100M,polars_lazy,filter,81.089,11.833,1.234
1,100M,polars_lazy,groupby,12.662,3.457,7.903
2,100M,dask,filter,111.920,11.198,0.894
3,100M,dask,groupby,17.161,6.123,5.828


,dataset_size,workload,framework,time_mean_s,memory_gb,rows_per_second_m
20,10M_highuid,filter,polars_eager,2.577,4.501,3.895
29,10M_highuid,groupby,dask,2.983,3.178,3.394
22,10M_highuid,join,polars_eager,4.539,5.600,2.208
23,10M_highuid,pipeline,polars_eager,6.786,5.626,1.480
4,10M_skewed,filter,polars_eager,1.891,4.517,5.290
9,10M_skewed,groupby,polars_lazy,0.624,1.149,16.042
6,10M_skewed,join,polars_eager,2.817,5.658,3.564
7,10M_skewed,pipeline,polars_eager,4.381,5.686,2.299


,stress_scenario,framework,workload,scope_status,failure_class,reason,n_runs,taxonomy_type
0,100M_real_like,pandas,filter/groupby/join/pipeline,omitted_by_design,practical_memory_envelope,Pandas 100M was excluded because it is outside...,NaN,scope_or_aborted_note
1,100M_real_like,polars_eager,filter/groupby/join/pipeline,attempted_aborted,system_shutdown,Polars eager 100M was attempted and caused the...,NaN,scope_or_aborted_note
2,100M_real_like,polars_lazy/dask,join,attempted_aborted,system_shutdown,100M join materialization/intermediate size wa...,NaN,scope_or_aborted_note
3,100M_real_like,polars_lazy/dask,pipeline,omitted_by_design,out_of_scope_due_to_join_pressure,Pipeline is excluded from 100M stress interpre...,NaN,scope_or_aborted_note


### Nhận xét

Các bảng synthetic stress cho thấy kết quả stress benchmark phụ thuộc rất mạnh vào từng stress scenario. Vì các scenario này được thiết kế để kiểm tra edge cases, chúng không nên được dùng để thay thế main real-data benchmark, mà nên được xem như bằng chứng bổ sung về robustness, memory pressure và workload sensitivity.

Ở nhóm `100M real-like`, chỉ có `polars_lazy` và `dask` được đưa vào reportable scope cho hai workload `filter` và `groupby`. Trong cả hai workload này, `polars_lazy` nhanh hơn `dask`. Với `filter`, `polars_lazy` chạy khoảng `81.09s`, nhanh hơn `dask` khoảng `111.92s`. Với `groupby`, `polars_lazy` chạy khoảng `12.66s`, nhanh hơn `dask` khoảng `17.16s`. Ngoài ra, `polars_lazy` cũng dùng ít memory hơn rõ rệt ở `groupby`, cho thấy Lazy execution có thể phù hợp với large-scale aggregation khi output cuối không quá lớn.

Ở nhóm `10M_skewed`, `polars_eager` đứng đầu ở `filter`, `join`, và `pipeline`, trong khi `polars_lazy` đứng đầu rất rõ ở `groupby`. Trường hợp `groupby 10M_skewed` có runtime khoảng `0.624s`, memory khoảng `1.149GB`, và throughput khoảng `16.04M rows/s`. Đây là một kết quả nổi bật, cho thấy Lazy có thể hưởng lợi mạnh khi workload aggregation gặp key distribution bị skew.

Ở nhóm `10M_highuid`, `polars_eager` đứng đầu ở `filter`, `join`, và `pipeline`, còn `dask` đứng đầu ở `groupby`. Điều này cho thấy high cardinality tạo áp lực khác với skew: khi số lượng unique key tăng cao, chi phí hash-table, key handling và memory overhead có thể làm thay đổi framework ranking. Trong scenario này, Dask có lợi ở `groupby`, trong khi Polars Eager vẫn ổn định hơn ở các workload còn lại.

Bảng failure/scope taxonomy cũng rất quan trọng để tránh diễn giải sai. `Pandas 100M` được loại khỏi scope vì vượt practical memory envelope. `Polars Eager 100M` từng được thử nhưng gây system shutdown. `Join 100M` với `polars_lazy/dask` cũng bị aborted do materialization hoặc intermediate size quá lớn. `Pipeline 100M` bị loại vì có join pressure, nên không nên so sánh trực tiếp với các workload reportable khác.

Tóm lại, synthetic stress benchmark cho thấy `polars_lazy` mạnh ở `100M real-like` cho `filter/groupby` và đặc biệt mạnh ở `groupby` khi dữ liệu skewed. `polars_eager` ổn định nhất ở nhiều workload 10M stress như `filter`, `join`, và `pipeline`. `Dask` có thể cạnh tranh mạnh ở `groupby`, đặc biệt trong high-cardinality scenario. Tuy nhiên, các kết quả này là stress-specific và chỉ nên dùng để hiểu sensitivity/failure behavior, không phải ranking tổng quát cho toàn bộ benchmark.

In [17]:
stress_report_ready = read_table("synthetic_stress_report_ready_summary.csv")
display(stress_report_ready)

,main_finding,evidence,caveat,report_ready_sentence
0,100M stress is feasible only for the scoped sc...,Polars lazy and Dask have successful `filter` ...,"Pandas, Polars eager, join, and pipeline are o...",The 100M synthetic stress test should be inter...
1,100M results should not be used as a scaling l...,The notebook shows 100M beside real 1M/10M/50M...,Synthetic and real datasets can differ in phys...,The 100M marker extends the visible scale rang...
2,10M stress sensitivity is evaluated within the...,Slowdown ratios match `10M_skewed` and `10M_hi...,Ratios are not comparable if the matching real...,Skew and high-cardinality effects are measured...
3,Omitted-by-design rows are not actual benchmar...,Failure taxonomy separates CSV-recorded failur...,Shutdown observations are still important prac...,The robustness analysis separates actual recor...
4,Polars lazy/eager stress behavior is workload-...,The notebook reports lazy speedup vs eager per...,Full lazy-vs-eager conclusions belong in the d...,"Within the 10M stress cases, Polars lazy and e..."


### Nhận xét

Bảng này đóng vai trò tổng hợp các kết luận quan trọng nhất từ synthetic stress benchmark và chuyển chúng thành các câu có thể dùng trực tiếp trong final report. Điểm quan trọng nhất là synthetic stress không được dùng như một benchmark ranking tổng quát, mà được dùng để đánh giá robustness, sensitivity và failure behavior trong các điều kiện dữ liệu cực đoan.

Kết quả `100M synthetic stress` chỉ nên được diễn giải trong phạm vi đã được scoped. Chỉ `polars_lazy` và `dask` có kết quả reportable cho `filter` và `groupby`. Các trường hợp như Pandas, Polars Eager, `join`, và `pipeline` không nằm trong cùng phạm vi so sánh vì bị loại khỏi scope, bị aborted hoặc tạo áp lực materialization quá lớn. Vì vậy, không nên nói rằng một framework “thắng toàn bộ 100M benchmark”; chỉ nên nói rằng framework đó hoàn thành các workload nằm trong reportable scope.

Bảng cũng nhấn mạnh rằng `100M synthetic` không phải là mốc scaling trực tiếp tiếp theo của real `1M/10M/50M`. Mặc dù nó giúp mở rộng phạm vi quan sát lên quy mô lớn hơn, dataset synthetic và real data có thể khác nhau về physical footprint, distribution, compression behavior và workload scope. Do đó, 100M result nên được đọc như stress evidence, không phải real row-scaling conclusion.

Với các stress cases ở `10M`, kết quả được so sánh trong cùng row count để kiểm tra ảnh hưởng của `skew` và `high cardinality`. Cách này hợp lý vì nó giữ row count cố định và thay đổi stress condition. Tuy nhiên, các slowdown hoặc speedup ratios vẫn cần được đọc cẩn thận, vì matching real baseline có thể khác nhau về data distribution và physical size.

Một điểm quan trọng khác là bảng tách riêng `omitted_by_design`, `attempted_aborted`, và recorded failures. Các dòng bị loại khỏi scope không nên được hiểu là failed benchmark run theo nghĩa CSV-recorded failure. Tuy vậy, các shutdown hoặc aborted observations vẫn có giá trị thực tế vì chúng chỉ ra practical limits của framework trong điều kiện memory hoặc materialization pressure rất cao.

Tóm lại, bảng này giúp final report diễn giải synthetic stress một cách an toàn: `100M` chỉ report trong scoped workloads, `10M skew/highuid` dùng để phân tích sensitivity, và các omitted/aborted cases được xem như robustness evidence thay vì ranking result. Điều này giúp tránh overclaim và giữ kết luận chính dựa trên real-data benchmark.

## 7. Validation — Synthetic Data Quality

Synthetic data được xem là **fit-for-purpose** cho mục tiêu `physical scaling` và `synthetic stress benchmark` vì nó đạt các validation targets chính. Mục tiêu của bước validation không phải là tạo bản sao hoàn hảo của Real Amazon Reviews dataset, mà là đảm bảo synthetic data giữ được các đặc điểm quan trọng ảnh hưởng đến benchmark behavior.

Ở nhóm real-like synthetic, schema và shape khớp với real sample. Memory footprint của synthetic data cao hơn real data khoảng `13.6%`, nhưng vẫn nằm trong mức hợp lý cho benchmark. Sai lệch này có thể chấp nhận được vì mục tiêu chính là giữ schema, row count, key behavior và các distribution quan trọng, chứ không phải tái tạo chính xác memory layout của real dataset.

Phân phối `text_len` khớp tốt với real data. Median gần như trùng nhau, các mốc p90 và p99 rất gần, và long-tail behavior được giữ lại. Đây là điểm quan trọng vì text length ảnh hưởng trực tiếp đến physical size, memory footprint, Parquet decoding cost và materialization cost.

Phân phối `rating` cũng gần như trùng với empirical rating distribution của real data. Synthetic data giữ được đặc điểm rating bị skew mạnh về các mức rating cao, nên workload `filter` vẫn có selectivity tương tự với real data. `helpful_vote` còn có sai lệch nhẹ ở tail trung-cao, nhưng biến này không phải key chính của benchmark nên không làm thay đổi mục tiêu validation tổng thể.

Cardinality validation cũng đạt yêu cầu. `user_id` cardinality ratio gần `0.99` so với real data, cho thấy mức độ đa dạng user được tái tạo tốt. `parent_asin` và `product_id` thấp hơn real data, nhưng vẫn nằm trong ngưỡng chấp nhận cho mục tiêu benchmark. Điều quan trọng là các key columns vẫn giữ được hành vi đủ thực tế để tạo áp lực cho `groupby`, `join`, và `pipeline`.

Với các stress profiles, synthetic data cũng đạt đúng mục tiêu thiết kế. `TIER2_SKEWED` tạo hot-key pressure rõ rệt, với top-1% `parent_asin` chiếm khoảng `95.3%` số dòng. Điều này phù hợp với mục tiêu kiểm tra sensitivity của framework dưới heavy skew. `TIER2_HIGH_UNIQUE` đạt unique ratio của `user_id` khoảng `50.0%`, nằm đúng trong range mục tiêu `45%–55%`, nên phù hợp để kiểm tra high-cardinality pressure.

Tóm lại, synthetic data đủ giống real data để dùng cho physical scaling và stress benchmark, đồng thời các stress profiles đủ khác real data theo hướng có chủ đích. Vì vậy, synthetic benchmark results có thể được dùng làm supporting evidence về memory pressure, skew sensitivity và high-cardinality behavior. Tuy nhiên, chúng vẫn nên được hiểu là synthetic approximation, không phải thay thế cho kết luận chính từ real-data benchmark.

## 8. Results — OS / Environment Comparison

Notebook `05_real_env_comparison.ipynb` nên được diễn giải là **environment comparison**, không phải pure OS comparison. Lý do là hai môi trường benchmark không chạy trên cùng phần cứng và không có cùng resource policy. Windows được chạy trên local machine với khoảng `16GB RAM`, trong khi Colab/Linux được chạy trên cloud runtime với khoảng `12GB RAM` và có cơ chế giới hạn hoặc termination riêng.

Vì vậy, phần này không nhằm kết luận hệ điều hành nào nhanh hơn tuyệt đối. Mục tiêu chính là kiểm tra xem các kết luận từ real-data benchmark có ổn định khi chuyển sang môi trường chạy khác hay không, và workload nào nhạy với resource limits hoặc runtime policy.

Với các dataset size có dữ liệu đầy đủ ở cả hai môi trường, cụ thể là `1M` và `10M`, framework ranking nhìn chung khá ổn định. Polars thường nằm trong nhóm nhanh nhất, đặc biệt `polars_eager` ở các workload như `filter`, `join`, và `pipeline`. Dask có thể dẫn đầu một số `groupby` case trên Colab/Linux, cho thấy partitioned execution vẫn có lợi trong một số aggregation workload. Pandas thường chậm hơn khi dữ liệu tăng, đặc biệt ở các workload có memory pressure cao hơn.

Runtime tuyệt đối không nghiêng một chiều cho mọi workload. `Filter` và `groupby` có xu hướng ổn định hơn giữa hai môi trường, trong khi `join` và `pipeline` nhạy hơn với environment và framework. Điều này hợp lý vì `join` và `pipeline` thường có output hoặc intermediate data lớn hơn, nên dễ bị ảnh hưởng bởi memory allocation, filesystem behavior, pagefile/swap, cache và final materialization.

Về memory, Windows local thường ghi nhận peak memory cao hơn Colab/Linux trong nhiều trường hợp, đặc biệt ở `join` và `pipeline`. Tuy nhiên, điều này không nên được diễn giải đơn giản là Windows dùng memory kém hơn. Sự khác biệt có thể đến từ allocator behavior, filesystem/cache behavior, measurement method, pagefile/swap, hoặc runtime termination policy của Colab/Linux.

Mốc `50M` cần được tách riêng khi diễn giải. Windows local hoàn thành được các workload `50M`, trong khi Colab/Linux không có successful run hoàn chỉnh ở mốc này. Kết quả này không nên được viết là “Windows nhanh hơn Linux”. Diễn giải đúng hơn là workload `50M` nhạy với resource limit: trong cấu hình được test, Windows local với khoảng `16GB RAM` hoàn thành được, còn Colab/Linux với khoảng `12GB RAM` và runtime policy chặt hơn thì không hoàn thành.

Tóm lại, environment comparison cho thấy các kết luận chính ở `1M` và `10M` tương đối ổn định, đặc biệt là vị thế mạnh của Polars. Tuy nhiên, ở workload lớn hoặc memory-intensive như `50M`, `join`, và `pipeline`, kết quả phụ thuộc mạnh vào môi trường chạy. Vì vậy, phần này nên được dùng để thảo luận về reproducibility và resource sensitivity, không phải để đưa ra kết luận tuyệt đối về hệ điều hành.

## 9. Workload Discussion

Workload-level analysis giúp giải thích vì sao framework ranking thay đổi giữa `filter`, `groupby`, `join`, và `pipeline`. Vì benchmark này là end-to-end, mỗi workload không chỉ đo một operator riêng lẻ, mà còn bao gồm chi phí đọc dữ liệu, execution strategy, memory allocation và final materialization.

**Filter** là workload mang tính scan-heavy. Performance phụ thuộc nhiều vào tốc độ đọc Parquet, khả năng predicate/projection pushdown, vectorized execution và chi phí materialize kết quả sau khi filter. Polars thường có lợi thế trong workload này nhờ columnar execution và optimizer. Đặc biệt, Polars Lazy có thể phát huy lợi thế khi optimizer loại bỏ các cột không cần thiết hoặc đẩy predicate xuống gần bước scan hơn. Tuy nhiên, nếu filter tạo output lớn, chi phí materialization vẫn có thể làm giảm lợi thế của Lazy.

**GroupBy** phụ thuộc vào aggregation strategy, hash table construction, key cardinality và key distribution. Khi key cardinality thấp hoặc partitioning thuận lợi, Dask có thể rất nhanh vì aggregation có thể được chia nhỏ theo partition trước khi combine kết quả. Tuy nhiên, khi dữ liệu có heavy skew hoặc high cardinality, groupby trở nên nhạy hơn với hot keys, memory layout, hash table size và partition imbalance. Vì vậy, groupby là workload có thể thay đổi ranking rõ giữa real row-scaling, physical scaling và synthetic stress benchmark.

**Join** là workload nặng nhất trong real row-scaling benchmark. Join không chỉ đo join algorithm, mà còn bao gồm key representation, build/probe cost, intermediate output size, final materialization và memory allocation. Kết quả cho thấy `polars_eager` rất mạnh ở join trong nhiều mốc dataset size, trong khi Dask và Pandas thường có overhead lớn hơn. Điều này cho thấy join là workload mà execution overhead và materialization cost có ảnh hưởng rất lớn đến end-to-end runtime.

**Pipeline** gồm nhiều bước xử lý nên có thể khuếch đại chi phí của từng stage. Pipeline thường bao gồm scan/read, filter, aggregation, join-like operations hoặc final transformation, nên runtime và memory phụ thuộc vào cả thứ tự operation lẫn kích thước intermediate data. Đây cũng là workload mà Lazy execution có cơ hội tối ưu whole-plan tốt nhất. Nếu optimizer có thể fuse, reorder operations, push filters earlier hoặc tránh intermediate materialization không cần thiết, Lazy có thể giảm runtime đáng kể. Ngược lại, nếu pipeline buộc phải materialize output lớn hoặc tạo join-like intermediate lớn, lợi thế của Lazy có thể giảm.

Tóm lại, workload discussion cho thấy framework performance không chỉ phụ thuộc vào dataset size, mà còn phụ thuộc mạnh vào cấu trúc workload. `Filter` kiểm tra scan efficiency, `groupby` kiểm tra aggregation và key behavior, `join` tạo áp lực lớn lên memory/materialization, còn `pipeline` kiểm tra khả năng tối ưu toàn bộ execution plan.

## 10. Polars Lazy vs Eager

Kết quả từ `07_lazy_vs_eager_polars.ipynb` cho thấy Polars Lazy **không luôn nhanh hơn** Polars Eager. Lazy có lợi thế rõ nhất ở các workload `groupby` và `pipeline` khi xét theo mean speedup, nhưng `join` nghiêng mạnh về Eager, còn `filter` gần như tương đương hoặc Eager là đủ cho workflow đơn giản.

Theo bảng recommendation, Lazy đạt mean speedup khoảng `1.88×` ở `groupby` và khoảng `1.70×` ở `pipeline`. Điều này cho thấy Lazy execution có thể phát huy hiệu quả khi workload có nhiều cơ hội tối ưu query plan, ví dụ projection pushdown, predicate pushdown, hoặc giảm intermediate materialization trong pipeline nhiều bước.

Tuy nhiên, lợi thế runtime của Lazy không đi kèm với lợi thế memory một cách nhất quán. Trong main real row-scaling benchmark, Lazy có mean memory ratio lớn hơn Eager ở tất cả workload chính trong bảng tổng hợp. Điều này nghĩa là Lazy không tự động giảm peak memory trong benchmark end-to-end này. Ở một số workload lớn, Lazy nhanh hơn nhưng vẫn dùng peak memory cao hơn.

`Join` là workload mà Eager có lợi thế rõ nhất. Lazy speedup ở join nhỏ hơn `1.0` khá rõ, nghĩa là Eager nhanh hơn Lazy. Điều này cho thấy chi phí optimization hoặc execution plan của Lazy không đủ để bù lại các chi phí chính của join như key handling, output materialization và memory allocation.

Recommendation thực tế:

- Dùng **Polars Eager** cho workflow đơn giản, `filter` trực tiếp, `join`, hoặc khi muốn runtime ổn định và peak memory thấp hơn.
- Dùng **Polars Lazy** khi query có nhiều bước, có cơ hội projection/predicate pushdown, hoặc khi `groupby`/`pipeline` có thể được tối ưu bằng whole-plan optimization.
- Ưu tiên **Polars Lazy** khi runtime là mục tiêu chính và môi trường có đủ memory headroom.
- Không xem Lazy là default thắng mọi case. Benchmark này đo end-to-end runtime, nên optimization overhead, scan/read, memory allocation và final materialization đều ảnh hưởng đến kết quả.

Tóm lại, Lazy phù hợp nhất với các workload lớn và có cấu trúc query phức tạp như `groupby` và `pipeline`. Eager vẫn là lựa chọn tốt hơn hoặc đủ tốt cho `join`, `filter` đơn giản và các workflow cần memory behavior ổn định hơn.

In [19]:
lazy_final = read_table("polars_lazy_vs_eager_final_answers.csv")
lazy_recommendations = read_table("polars_lazy_vs_eager_recommendations.csv")
lazy_speedup = read_table("polars_lazy_speedup_by_workload.csv")
lazy_memory = read_table("polars_lazy_memory_ratio_by_workload.csv")

display(lazy_final)
display(lazy_recommendations.round(3))
display(lazy_speedup.round(3))
display(lazy_memory.round(3))

,question,answer_from_results
0,Is Lazy consistently faster?,No; see speedup_by_size for workload-size exce...
1,Workload with strongest Lazy benefit,groupby
2,Workloads where Eager is faster or similar,"filter, join"
3,Does Lazy reduce memory pressure?,Mixed; see memory ratio tables.
4,Does behavior change with dataset size?,Yes
5,Recommendation source,Use recommendations table; it is computed from...


,workload,mean_lazy_speedup,overall_faster_mode,mean_lazy_memory_ratio,overall_lower_memory_mode,recommended_mode,reason,memory_note
0,filter,1.029,Similar,1.079,Eager,Eager is enough,mean speedup=1.03x; memory ratio=1.08,Eager uses less peak memory on average.
1,groupby,1.877,Lazy,1.329,Eager,Lazy if runtime matters,mean speedup=1.88x; memory ratio=1.33,Eager uses less peak memory on average.
2,join,0.370,Eager,1.241,Eager,Eager,mean speedup=0.37x; memory ratio=1.24,Eager uses less peak memory on average.
3,pipeline,1.698,Lazy,1.225,Eager,Lazy if runtime matters,mean speedup=1.70x; memory ratio=1.22,Eager uses less peak memory on average.


,workload,mean_lazy_speedup,min_lazy_speedup,max_lazy_speedup,n_dataset_sizes,overall_faster_mode
0,groupby,1.877,0.688,4.245,3,Lazy
1,pipeline,1.698,0.226,4.230,3,Lazy
2,filter,1.029,0.493,1.739,3,Similar
3,join,0.370,0.171,0.567,3,Eager


,workload,mean_lazy_memory_ratio,min_lazy_memory_ratio,max_lazy_memory_ratio,n_dataset_sizes,overall_lower_memory_mode
0,filter,1.079,0.768,1.569,3,Eager
1,pipeline,1.225,1.077,1.453,3,Eager
2,join,1.241,0.968,1.690,3,Eager
3,groupby,1.329,0.920,1.711,3,Eager


### Nhận xét

Các bảng tổng hợp xác nhận rằng Polars Lazy **không consistently faster** so với Polars Eager. Lazy có lợi thế rõ ở một số workload, nhưng không thắng trên toàn bộ workload và dataset size. Kết quả cuối cùng trả lời trực tiếp rằng Lazy không luôn nhanh hơn; cần xem theo từng workload và từng dataset size.

Theo bảng recommendation, `groupby` là workload có lợi nhất cho Lazy, với `mean_lazy_speedup ≈ 1.88x`. `Pipeline` cũng có lợi rõ cho Lazy, với `mean_lazy_speedup ≈ 1.70x`. Tuy nhiên, cả hai workload này đều có `mean_lazy_memory_ratio > 1.0`, nghĩa là Lazy nhanh hơn về runtime nhưng dùng nhiều peak memory hơn Eager khi xét trung bình.

`Filter` có `mean_lazy_speedup ≈ 1.03x`, gần như tương đương Eager. Vì memory ratio của Lazy khoảng `1.08`, Eager dùng ít peak memory hơn trung bình. Do đó, với các filter workflow đơn giản, `Eager is enough` là recommendation hợp lý.

`Join` là workload nghiêng mạnh nhất về Eager. Lazy chỉ đạt `mean_lazy_speedup ≈ 0.37x`, đồng thời memory ratio khoảng `1.24`, nghĩa là Lazy vừa chậm hơn vừa dùng nhiều peak memory hơn. Vì vậy, recommendation cho `join` là dùng `Eager`.

Bảng speedup theo workload cũng cho thấy dataset size ảnh hưởng mạnh đến kết quả. `Groupby` và `pipeline` có max speedup hơn `4x`, nhưng min speedup vẫn nhỏ hơn `1.0`, nghĩa là Lazy không thắng ở mọi dataset size. Điều này cho thấy lợi thế của Lazy chủ yếu xuất hiện ở dataset lớn, không phải ở mọi quy mô.

Tóm lại, Lazy phù hợp nhất khi runtime là ưu tiên chính và workload có nhiều cơ hội tối ưu query plan, đặc biệt là `groupby` và `pipeline` lớn. Eager phù hợp hơn cho `join`, `filter` đơn giản, hoặc khi cần memory behavior ổn định hơn. Vì benchmark này là end-to-end, quyết định chọn Lazy hay Eager nên dựa trên cả runtime, peak memory và workload type, không chỉ dựa vào việc Lazy có optimizer.

## 11. Limitations

Benchmark này là **single-machine benchmark**. Vì vậy, kết quả không nên được mở rộng trực tiếp thành kết luận cho distributed cluster, production lakehouse, GPU dataframe engine hoặc database system. Các framework được đánh giá trong điều kiện workload và môi trường cụ thể của project này, không phải trong mọi cấu hình production có thể có.

Kết quả benchmark nhạy với hardware và execution environment. Các yếu tố như CPU, RAM, storage type, filesystem, OS cache, pagefile/swap, background load, framework version và runtime policy đều có thể ảnh hưởng đến runtime và peak memory. Đặc biệt, so sánh giữa Windows local và Colab/Linux nên được hiểu là **environment comparison**, không phải pure OS comparison, vì phần cứng, RAM và resource policy không giống nhau.

Synthetic data chỉ là một approximation của real data. Sau bước validation, synthetic data được xem là đủ phù hợp cho mục tiêu `physical scaling` và `synthetic stress benchmark`, nhưng nó không thể thay thế hoàn toàn Real Amazon Reviews dataset. Ngoài ra, một số synthetic profiles còn cố ý làm dữ liệu khác real data, ví dụ tạo heavy skew hoặc high cardinality, để kiểm tra stress behavior. Vì vậy, synthetic results nên được diễn giải như supporting evidence, không phải main real-world conclusion.

Benchmark này đo **end-to-end runtime**, không đo pure operator speed. Thời gian đo bao gồm Parquet scan/read, decoding, query planning, optimization hoặc scheduling, memory allocation, execution và final result materialization. Do đó, không nên trừ I/O time hoặc memory effect một cách cơ học để suy ra engine time, vì các thành phần này có thể overlap hoặc tương tác với nhau trong quá trình thực thi.

Các workload cũng có output size và materialization behavior khác nhau. `.collect()` trong Polars Lazy và `.compute()` trong Dask có thể materialize kết quả cuối trong memory. Với các workload có output lớn như `join` hoặc `pipeline`, final materialization và intermediate data có thể tạo memory pressure mạnh, làm tăng runtime hoặc khiến workload không hoàn thành trong môi trường bị giới hạn tài nguyên.

Ngoài ra, benchmark sử dụng một tập workload cố định gồm `filter`, `groupby`, `join`, và `pipeline`. Các kết luận có thể không áp dụng trực tiếp cho workload khác như window functions, complex string processing, nested data, machine learning preprocessing hoặc streaming sink-to-disk workflows. Những trường hợp này cần benchmark riêng nếu muốn kết luận chắc chắn.

Tóm lại, kết quả trong report nên được hiểu là practical end-to-end performance cho các workload và môi trường đã được kiểm tra. Chúng hữu ích để so sánh framework trong bối cảnh project này, nhưng không nên được xem là kết luận tuyệt đối cho mọi dataset, mọi workload hoặc mọi môi trường triển khai.

## 12. Conclusion

Benchmark này nên được hiểu là **practical end-to-end dataframe performance benchmark** trên large-scale tabular review data. Kết quả phản ánh toàn bộ quá trình xử lý workload, bao gồm đọc dữ liệu, execution, optimization hoặc scheduling, memory allocation và final result materialization.

**Pandas** phù hợp với small/simple workloads và là baseline dễ hiểu để so sánh. Tuy nhiên, Pandas không phải lựa chọn tốt nhất khi row count, memory footprint hoặc intermediate output tăng mạnh. Trong các benchmark lớn hơn, Pandas thường có runtime cao hơn và throughput thấp hơn rõ rệt so với Polars và Dask.

**Polars** là lựa chọn mạnh nhất cho single-machine high-performance analytics trong benchmark này. `polars_eager` đặc biệt tốt ở nhiều workload trực tiếp như `filter`, `join`, và `pipeline`, nhất là ở `10M` và trong nhiều physical scaling cases. `polars_lazy` rất hữu ích khi optimizer có thể tận dụng whole-plan optimization, đặc biệt ở `groupby`, `pipeline`, và một số workload lớn tại `50M`.

**Dask** hữu ích khi dữ liệu lớn, partitioned, hoặc cần out-of-core style execution. Dask rất cạnh tranh ở một số `groupby` workload, đặc biệt khi aggregation có thể chia nhỏ theo partition. Tuy nhiên, scheduling overhead, task graph execution và partition coordination khiến Dask không luôn nhanh nhất trong end-to-end benchmark, đặc biệt với `join` và `pipeline`.

**Polars Lazy** không phải default thắng mọi case. Lazy có thể nhanh hơn nhiều ở `groupby` và `pipeline`, nhưng Eager tốt hơn hoặc tương đương ở `filter` và `join`. Ngoài ra, trong main Lazy-vs-Eager benchmark, Eager thường có peak memory thấp hơn trong bảng tổng hợp. Vì vậy, lựa chọn giữa Lazy và Eager nên dựa trên workload shape, dataset size và trade-off giữa runtime với memory.

Kết luận quan trọng nhất là kết quả cần được đọc theo từng benchmark group và từng workload. `Real Data Row Scaling`, `Physical Scaling`, `Synthetic Stress`, và `OS / Environment Comparison` trả lời các câu hỏi khác nhau. Việc tách riêng các nhóm này giúp benchmark phản ánh đúng performance thực tế, thay vì tạo một ranking đơn giản nhưng dễ gây hiểu nhầm.

Không có framework nào thắng tuyệt đối dưới mọi workload, dataset size và execution environment. Lựa chọn thực tế phụ thuộc vào workload shape, data size, memory pressure, data distribution, execution environment và mục tiêu ưu tiên là runtime, memory hay reliability.